# Healthcare Operations SQL Validation Notebook

This notebook supports the validation expectations for AI-assisted SQL work. Use it before accepting an AI-generated query as correct. It helps you check schema fit, row counts, date ranges, status values, joins, filters, and expected outputs.

## Validation workflow

Use these cells in order when reviewing AI-generated SQL:

1. Confirm the database connection and available tables.
2. Inspect schemas and row counts.
3. Check fixed reporting dates and valid status values.
4. Check joins before trusting multi-table results.
5. Add filters one at a time.
6. Run expected-output checks or sanity checks.

In [41]:
import sqlite3
import pandas as pd

DB_FILE = "healthcare_operations.db"
conn = sqlite3.connect(DB_FILE)
print(f"Connected to {DB_FILE}")

Connected to healthcare_operations.db


## 1. Table list check

Start by confirming that the database contains the tables your query expects. This helps catch prompts or SQL that assume the wrong database setup.

In [42]:
tables = pd.read_sql("""
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
  AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""", conn)
tables

,table_name
0,appointments
1,clinics
2,patients
3,providers
4,reporting_context


## 2. Schema inspection

Use the schema as the source of truth. Any table or column that does not appear here should be treated as a possible hallucination.

In [43]:
for table_name in tables['table_name']:
    print(f"\n--- {table_name} ---")
    display(pd.read_sql(f"PRAGMA table_info({table_name});", conn))


--- appointments ---


,cid,name,type,notnull,dflt_value,pk
0,0,appointment_id,INTEGER,0,None,1
1,1,patient_id,INTEGER,1,None,0
2,2,provider_id,INTEGER,1,None,0
3,3,clinic_id,INTEGER,1,None,0
4,4,appointment_date,TEXT,1,None,0
5,5,appointment_status,TEXT,1,None,0
6,6,appointment_type,TEXT,1,None,0



--- clinics ---


,cid,name,type,notnull,dflt_value,pk
0,0,clinic_id,INTEGER,0,None,1
1,1,clinic_name,TEXT,1,None,0
2,2,city,TEXT,1,None,0
3,3,region,TEXT,1,None,0



--- patients ---


,cid,name,type,notnull,dflt_value,pk
0,0,patient_id,INTEGER,0,None,1
1,1,patient_name,TEXT,0,None,0
2,2,date_of_birth,TEXT,0,None,0
3,3,city,TEXT,0,None,0
4,4,insurance_type,TEXT,0,None,0



--- providers ---


,cid,name,type,notnull,dflt_value,pk
0,0,provider_id,INTEGER,0,None,1
1,1,provider_name,TEXT,0,None,0
2,2,specialty,TEXT,0,None,0



--- reporting_context ---


,cid,name,type,notnull,dflt_value,pk
0,0,context_id,INTEGER,0,None,1
1,1,reporting_date,TEXT,1,None,0
2,2,data_start_date,TEXT,1,None,0
3,3,data_end_date,TEXT,1,None,0
4,4,recent_30_day_start,TEXT,1,None,0
5,5,recent_60_day_start,TEXT,1,None,0
6,6,recent_3_month_start,TEXT,1,None,0
7,7,recent_6_month_start,TEXT,1,None,0
8,8,primary_date_column,TEXT,1,None,0
9,9,note,TEXT,0,None,0


## 3. Row counts

Check row counts before and after writing complex queries. If a source table has zero rows, a query that returns no rows may be behaving correctly.

In [44]:
row_counts = []
for table_name in tables['table_name']:
    count = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table_name};", conn).iloc[0, 0]
    row_counts.append({'table_name': table_name, 'row_count': count})
pd.DataFrame(row_counts)

,table_name,row_count
0,appointments,120
1,clinics,3
2,patients,50
3,providers,10
4,reporting_context,1


## 4. Reporting context and min/max dates

Use the fixed reporting context instead of live-date logic such as `DATE('now')`. This prevents fictional historical datasets from returning empty or misleading results.

In [45]:
pd.read_sql("SELECT * FROM reporting_context;", conn)

,context_id,reporting_date,data_start_date,data_end_date,recent_30_day_start,recent_60_day_start,recent_3_month_start,recent_6_month_start,primary_date_column,note
0,1,2025-06-29,2025-01-06,2025-06-29,2025-05-30,2025-04-30,2025-03-29,2024-12-29,appointments.appointment_date,Use reporting_date instead of date('now') beca...


In [46]:
pd.read_sql("SELECT * FROM dataset_date_range;", conn)

,table_name,date_column,min_date,max_date,row_count
0,appointments,appointment_date,2025-01-06,2025-06-29,120


## 5. Valid status and category values

Check exact spelling and capitalization before writing filters. Small mismatches such as `no-show` vs. `no_show` or `canceled` vs. `Canceled` can change results.

In [47]:
pd.read_sql("SELECT * FROM status_value_counts ORDER BY table_name, status_column, row_count DESC;", conn)

,table_name,status_column,status_value,row_count
0,appointments,appointment_status,completed,87
1,appointments,appointment_status,no_show,23
2,appointments,appointment_status,canceled,10
3,appointments,appointment_type,follow_up,36
4,appointments,appointment_type,consult,32
5,appointments,appointment_type,checkup,29
6,appointments,appointment_type,screening,23


## 6. Join coverage checks

Check whether appointment records match valid patients, providers, and clinics before trusting a multi-table query.

In [48]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_appointments,
    COUNT(p.patient_id) AS appointments_with_matching_patient,
    COUNT(pr.provider_id) AS appointments_with_matching_provider,
    COUNT(c.clinic_id) AS appointments_with_matching_clinic
FROM appointments AS a
LEFT JOIN patients AS p
    ON a.patient_id = p.patient_id
LEFT JOIN providers AS pr
    ON a.provider_id = pr.provider_id
LEFT JOIN clinics AS c
    ON a.clinic_id = c.clinic_id;
""", conn)

,total_appointments,appointments_with_matching_patient,appointments_with_matching_provider,appointments_with_matching_clinic
0,120,120,120,120


## 7. Row-count-before-and-after join checks

These joins should preserve appointment-level rows when each appointment links to one patient, provider, and clinic.

In [49]:
pd.read_sql("""
SELECT 'appointments only' AS query_step, COUNT(*) AS row_count FROM appointments
UNION ALL
SELECT 'appointments + patients', COUNT(*)
FROM appointments AS a
JOIN patients AS p
    ON a.patient_id = p.patient_id
UNION ALL
SELECT 'appointments + patients + providers + clinics', COUNT(*)
FROM appointments AS a
JOIN patients AS p
    ON a.patient_id = p.patient_id
JOIN providers AS pr
    ON a.provider_id = pr.provider_id
JOIN clinics AS c
    ON a.clinic_id = c.clinic_id;
""", conn)

,query_step,row_count
0,appointments only,120
1,appointments + patients,120
2,appointments + patients + providers + clinics,120


## 8. Filter-by-filter debugging

Use exact status values from the database. In this dataset, missed appointments are stored as `no_show`.

In [50]:
pd.read_sql("""
SELECT 'all appointments' AS step, COUNT(*) AS row_count FROM appointments
UNION ALL
SELECT 'last 3 months', COUNT(*)
FROM appointments
WHERE appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
UNION ALL
SELECT 'last 3 months and no_show', COUNT(*)
FROM appointments
WHERE appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND appointment_status = 'no_show';
""", conn)

,step,row_count
0,all appointments,120
1,last 3 months,58
2,last 3 months and no_show,15


## 9. Expected-output check: repeat no-show patients

This validates a common healthcare lab pattern: patients with repeated no-shows in the anchored recent period.

In [51]:
repeat_no_show_patients = pd.read_sql("""
SELECT
    p.patient_id,
    p.patient_name,
    COUNT(a.appointment_id) AS no_show_count
FROM patients AS p
JOIN appointments AS a
    ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (
    SELECT recent_3_month_start
    FROM reporting_context
)
  AND a.appointment_status = 'no_show'
GROUP BY p.patient_id, p.patient_name
HAVING COUNT(a.appointment_id) >= 2
ORDER BY no_show_count DESC, p.patient_name;
""", conn)

expected_columns = ['patient_id', 'patient_name', 'no_show_count']
assert list(repeat_no_show_patients.columns) == expected_columns
repeat_no_show_patients

,patient_id,patient_name,no_show_count
0,28,Brandon Carter,2
1,39,Maya Stewart,2
2,17,Quinn Lewis,2
3,46,Thomas Bell,2


## Planning Statement

The business question I am answering is: Which patients have repeated no-show appointments?

This question matters because repeated no-shows can reduce clinic efficiency, leave appointment slots unused, and may indicate patients who need additional follow-up, reminders, transportation support, or scheduling assistance.

For this analysis, I define a high-risk patient as a patient with two or more appointments where `appointment_status = 'no_show'`.

The expected SQL output is one row per high-risk patient, including patient ID, patient name, city, insurance type, number of no-show appointments, first no-show date, most recent no-show date, and clinic/provider context. The primary scope is the full available dataset; the anchored recent three-month check is used only as a comparison.

For this workflow, a reliable result must use verified tables and columns, preserve appointment rows through valid joins, use the exact `no_show` value, apply the two-or-more threshold correctly, return one row per patient, match the stated reporting scope, and disclose assumptions and limitations that could affect decisions.

## Schema and Query Requirements

The query needs the `patients`, `appointments`, `clinics`, and `providers` tables.

Required columns include:

- `patients.patient_id`
- `patients.patient_name`
- `patients.city`
- `patients.insurance_type`
- `appointments.appointment_id`
- `appointments.patient_id`
- `appointments.provider_id`
- `appointments.clinic_id`
- `appointments.appointment_date`
- `appointments.appointment_status`
- `clinics.clinic_id`
- `clinics.clinic_name`
- `clinics.region`
- `providers.provider_id`
- `providers.provider_name`
- `providers.specialty`

Required joins:

- `patients.patient_id = appointments.patient_id`
- `clinics.clinic_id = appointments.clinic_id`
- `providers.provider_id = appointments.provider_id`

The query should filter to `appointment_status = 'no_show'`, group by patient, count no-show appointments, include only patients with two or more no-shows, and sort by no-show count descending.

## Validation Checklist

The existing validation workflow is evaluated against these criteria before accepting the revised result:

- **Schema accuracy:** Confirm that every referenced table and column exists in the database; treat unverified fields as possible hallucinations.
- **Join correctness:** Confirm the patient, clinic, and provider relationships and verify that joins preserve the expected appointment-level row count.
- **Filter accuracy:** Confirm that `appointment_status = 'no_show'` is an actual value and that the reporting period is explicit rather than inferred from the current date.
- **Aggregation and grouping:** Confirm one row per patient, correct no-show counts, correct minimum and maximum dates, and `HAVING` logic requiring at least two no-shows.
- **Business alignment:** Confirm that the output supports follow-up planning and that `high-risk` is understood as an operational threshold, not a clinical diagnosis.
- **Completeness:** Confirm that required patient attributes, dates, counts, and context fields are present and that no important constraint is silently omitted.
- **Risk and edge cases:** Check for hallucinated fields, missing join matches, date-scope mismatches, duplicate-count risks, and loss of relationships when values are concatenated.
- **Output validation:** Confirm that the result is decision-ready, clearly labeled by scope, and consistent with the persisted validation evidence.

## Initial Structured Prompt

Task:
Generate a SQLite query to identify patients with repeated no-show appointments in a healthcare operations database. A repeated no-show patient is a patient who has at least 2 appointments with `appointment_status = 'no_show'`.

Schema:
Use the following tables and columns:
- patients: patient_id, patient_name, date_of_birth, city, insurance_type
- appointments: appointment_id, patient_id, provider_id, clinic_id, appointment_date, appointment_status, appointment_type

Join patients to appointments using `patients.patient_id = appointments.patient_id`.

Constraints:
- Use SQLite syntax.
- Only count appointments where `appointment_status = 'no_show'`.
- Group results by patient.
- Include only patients with 2 or more no-show appointments.
- Do not use tables or columns not listed above.

Output:
Return patient_id, patient_name, city, insurance_type, no_show_count, first_no_show_date, and most_recent_no_show_date. Sort by no_show_count descending.

## Initial AI-Generated SQL

```sql
SELECT
    p.patient_id,
    p.patient_name,
    p.city,
    p.insurance_type,
    COUNT(a.appointment_id) AS no_show_count,
    MIN(a.appointment_date) AS first_no_show_date,
    MAX(a.appointment_date) AS most_recent_no_show_date
FROM patients AS p
JOIN appointments AS a
    ON p.patient_id = a.patient_id
WHERE a.appointment_status = 'no_show'
GROUP BY
    p.patient_id,
    p.patient_name,
    p.city,
    p.insurance_type
HAVING COUNT(a.appointment_id) >= 2
ORDER BY no_show_count DESC;

## SQL Review Notes

The initial query uses valid `patients` and `appointments` tables, the correct patient join, the exact `no_show` filter, patient-level grouping, date aggregation, and a threshold of at least two no-shows. Its patient-level logic is therefore sound for the full available dataset.

The main omission is clinic and provider context. That omission matters because the business scenario includes clinic efficiency and follow-up planning; patient counts alone do not show where outreach or scheduling support may be concentrated. The revised query adds this context and a secondary sort by the most recent no-show date while preserving one row per patient.

The review also identified two risks. First, the recent three-month validation check and the unfiltered revised query use different scopes, so their patient totals must not be compared as if they were the same analysis. Second, `GROUP_CONCAT(DISTINCT ...)` summarizes values but does not preserve exact provider-to-specialty or clinic-to-appointment pairings.

These findings came from comparing the SQL with the verified schema, join coverage, status values, persisted outputs, and stated business question.

## Revised Prompt

Task:
Generate a SQLite query that identifies high-risk patients with repeated no-show appointments and provides useful clinic/provider context for follow-up planning. A high-risk patient is defined as a patient with 2 or more appointments where `appointment_status = 'no_show'`.

Schema:
Use only these tables and columns:
- patients: patient_id, patient_name, city, insurance_type
- appointments: appointment_id, patient_id, provider_id, clinic_id, appointment_date, appointment_status
- clinics: clinic_id, clinic_name, region
- providers: provider_id, provider_name, specialty

Required joins:
- `patients.patient_id = appointments.patient_id`
- `clinics.clinic_id = appointments.clinic_id`
- `providers.provider_id = appointments.provider_id`

Constraints:
- Use SQLite syntax.
- Only count appointments where `appointments.appointment_status = 'no_show'`.
- Return one row per patient.
- Include only patients with 2 or more no-show appointments.
- Use `GROUP_CONCAT(DISTINCT ...)` to summarize associated clinic names, regions, provider names, and specialties.
- Do not invent tables, columns, statuses, or relationships.

Output:
Return patient_id, patient_name, city, insurance_type, no_show_count, first_no_show_date, most_recent_no_show_date, clinics_involved, regions_involved, providers_involved, and specialties_involved. Sort by no_show_count descending, then most_recent_no_show_date descending.

## Four-Step Prompt Chain

The workflow is documented as four linked prompts. This review records the prompt structure while preserving the existing revised SQL and its saved output. No new SQL is generated during this audit.

### Prompt 1: Clarify and restate the task

Restate the healthcare operations question in plain language. Define a repeated no-show patient as a patient with at least two appointments whose exact status is `no_show`. State whether the analysis uses the full available dataset or a fixed reporting period, identify the intended grain as one row per patient, and list the decision the output should support. Identify any ambiguity before writing SQL.

### Prompt 2: Generate SQL

Using only the verified SQLite schema, generate a query that counts qualifying no-show appointments, groups by patient, applies the two-or-more threshold, returns the required patient and date fields, and includes clinic/provider context without creating multiple rows per patient. Use the stated reporting scope and do not invent tables, columns, statuses, relationships, or live-date assumptions.

### Prompt 3: Review and validate the SQL

Review the proposed SQL against the verified schema and validation checklist. Check tables, columns, joins, row preservation, exact status filters, reporting dates, grouping, calculations, threshold logic, output grain, hallucinations, assumptions, and edge cases. List every issue, explain why it matters to healthcare decision-making, and distinguish confirmed facts from assumptions.

### Prompt 4: Refine the output

Revise the SQL only in response to confirmed review findings. Preserve one row per patient, retain the exact no-show definition and stated scope, add only verified context, and explain how each revision addresses a documented issue. Return the revised SQL and a concise limitations note.

In [52]:
revised_query = """
SELECT
    p.patient_id,
    p.patient_name,
    p.city,
    p.insurance_type,
    COUNT(a.appointment_id) AS no_show_count,
    MIN(a.appointment_date) AS first_no_show_date,
    MAX(a.appointment_date) AS most_recent_no_show_date,
    GROUP_CONCAT(DISTINCT c.clinic_name) AS clinics_involved,
    GROUP_CONCAT(DISTINCT c.region) AS regions_involved,
    GROUP_CONCAT(DISTINCT pr.provider_name) AS providers_involved,
    GROUP_CONCAT(DISTINCT pr.specialty) AS specialties_involved
FROM patients AS p
JOIN appointments AS a
    ON p.patient_id = a.patient_id
JOIN clinics AS c
    ON a.clinic_id = c.clinic_id
JOIN providers AS pr
    ON a.provider_id = pr.provider_id
WHERE a.appointment_status = 'no_show'
GROUP BY
    p.patient_id,
    p.patient_name,
    p.city,
    p.insurance_type
HAVING COUNT(a.appointment_id) >= 2
ORDER BY
    no_show_count DESC,
    most_recent_no_show_date DESC;
"""

pd.read_sql(revised_query, conn)

,patient_id,patient_name,city,insurance_type,no_show_count,first_no_show_date,most_recent_no_show_date,clinics_involved,regions_involved,providers_involved,specialties_involved
0,46,Thomas Bell,Springville,Medicare,2,2025-04-17,2025-05-17,Central City Medical Group,Central,Dr. Singh,Neurology
1,28,Brandon Carter,Lakewood,HMO,2,2025-04-10,2025-05-10,Central City Medical Group,Central,Dr. Wright,Psychiatry
2,39,Maya Stewart,Springville,Medicaid,2,2025-04-02,2025-04-28,Happy Valley Community Clinic,East,Dr. Chen,Pediatrics
3,17,Quinn Lewis,Happy Valley,Medicaid,2,2025-04-01,2025-04-25,Springville Family Health Center,North,"Dr. Lopez,Dr. Chen","Orthopedics,Pediatrics"
4,8,Hannah Davis,Happy Valley,HMO,2,2025-02-04,2025-03-06,Happy Valley Community Clinic,East,Dr. Nguyen,Primary Care
5,3,Carlos Rivera,Central City,Medicare,2,2025-01-14,2025-02-14,Central City Medical Group,Central,Dr. Lopez,Orthopedics


## Accuracy Review: Findings and Risks

The validation results support the structural correctness of the workflow. The database contains the expected tables and columns, and all 120 appointment records matched a patient, provider, and clinic. The `no_show` status is valid, and the grouping and `HAVING` logic correctly identifies patients with at least two no-show appointments.

The initial query is logically sound for patient-level no-show counts, dates, and thresholding, but it does not include clinic or provider context. The revised query adds that context and still returns one row per patient. Its `GROUP_CONCAT(DISTINCT ...)` fields are useful summaries, but they do not preserve exact provider-to-specialty or clinic-to-appointment pairings.

A scope difference must be made explicit. The anchored recent three-month validation check covers 15 no-show appointments and returns 4 patients with repeated no-shows. The revised query has no date filter and covers the full available dataset, so it returns 6 patients. The revised result should therefore be labeled as a full-dataset analysis rather than compared directly with the recent-period result.

The term `high-risk` is an operational label defined here as two or more no-shows; it is not a clinical risk determination. Overall, the SQL is schema-valid and logically aligned with the stated full-dataset question, with the reporting-period distinction and the summarized context fields remaining important limitations.

## Testing Evidence

The revised SQL ran successfully in SQLite, and no executed notebook cell contains a recorded error. The database validation showed the expected tables and row counts: `patients` 50, `appointments` 120, `providers` 10, `clinics` 3, and `reporting_context` 1.

The status check confirmed that `appointment_status = 'no_show'` is valid, with 23 total no-show appointments in the dataset. All 120 appointment records matched a patient, provider, and clinic, and the join row-count checks remained at 120 after adding those relationships.

### Before-versus-after comparison

| Version | Scope and result | What it provides | Validation conclusion |
|---|---|---|---|
| Initial query | Full available dataset; patient-level repeated no-show population | Patient identity, insurance, count, and first/most recent no-show dates | Correct core logic, but missing clinic/provider context |
| Revised query | Full available dataset; 6 patients with two or more no-shows | Initial fields plus summarized clinics, regions, providers, and specialties | Preserves one row per patient and adds useful operational context |
| Recent-period check | Anchored recent three months; 15 no-show appointments and 4 repeated-no-show patients | Narrow comparison population | Valid check, but not directly comparable to the full-dataset result |

The revised result meets the stated full-dataset business goal for identifying repeated no-shows and supporting follow-up planning. Remaining limitations are that `high-risk` is an operational threshold rather than a clinical risk determination, the reporting period must remain clearly labeled, and `GROUP_CONCAT(DISTINCT ...)` does not preserve exact provider-to-specialty or clinic-to-appointment pairings.

## Reflection

### Most important issue found

The most important issue was the difference between the recent three-month validation scope and the full-dataset scope of the revised query. Without labeling the scopes, the 4-patient and 6-patient results could be misread as contradictory or used to make an invalid trend comparison.

### How the prompt chain improved the workflow

The prompt chain made the business definition, output grain, schema restrictions, review criteria, and refinement steps explicit. It also encouraged reviewing joins, filters, aggregation, and assumptions before treating a successful SQL execution as proof of correctness.

### How validation changed my thinking

Validation changed my thinking by showing that a query can run without errors and still be incomplete or misleading. I now distinguish syntax success from schema validity, logical correctness, business alignment, reporting-scope consistency, and decision-readiness.

### Application to real-world data workflows

In a real healthcare workflow, this process would create an audit trail before outreach or staffing decisions are made: confirm the source data, test joins and filters, document definitions and scope, compare outputs, and disclose limitations. This reduces the risk of directing resources based on hallucinated fields, incorrect counts, or an operational label being mistaken for a clinical conclusion.

In [53]:
conn.close()
print("Database connection closed.")

Database connection closed.
